# Confidential Radiology AI — Secure Chest X-ray Inference on IBM Z
**Team Revolution · IBM Z Datathon 2026 · Theme: AI Secured** — research prototype, **not a diagnostic tool**.

A chest X-ray is encrypted before storage. A separate **attestation service + key broker** releases the study key only to a guest whose **code and model hashes** match the approved policy. The guest decrypts, runs a frozen ResNet-50, **wipes** the image buffer and returns only a **fixed, signed JSON** (probability, model hash, study hash). Requests are rate-limited and every decision goes to a hash-chained audit log without image content.

| Real here | Simulated / not proven |
|---|---|
| AES-256-GCM (PyCryptodome), per-study keys bound to the study id | **IBM Secure Execution**: this server is not an SE guest; the "guest" is a separate OS process |
| KEK held only by a separate broker process; X25519 + HKDF key transfer to the guest | Attestation measurement is **self-reported** by the guest (on SE the Ultravisor signs it) |
| Ed25519-signed, 60 s attestation tokens; single-use nonces; signed responses | Crypto Express HSM: the KEK is a chmod-600 file |
| Strict response schema checked on both sides; hash-chained audit; rate limiting | Guest egress block is in-process, not a firewall |
| Real ResNet-50 inference on the IBM z17 CPU | Host exclusion and memory encryption are **not demonstrated** |

Run the cells top to bottom. Sections: **0** environment · **1** setup · **2** model + policy · **3** broker · **4** live demo · **5** security tests · **6** secure-vs-clear + latency + Telum · **7** RSNA data and fine-tuning · **8** results + scope.

## 0 · Environment check (read-only)

In [1]:
import os, sys, platform, importlib, importlib.util, shutil, subprocess
os.environ["TORCHDYNAMO_DISABLE"] = "1"          # torch.compile is not used (unreliable on s390x)
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
env = {
    "machine": platform.machine(), "python": sys.version.split()[0], "cpus": os.cpu_count(),
    "z_machine_type": sh("grep -m1 -o 'machine = [0-9]*' /proc/cpuinfo"),
    "nnpa_facility": "nnpa" in sh("grep -m1 features /proc/cpuinfo"),
    "secure_execution_guest": sh("cat /sys/firmware/uv/prot_virt_guest 2>/dev/null") or "no (/sys/firmware/uv absent)",
    "crypto_express": os.path.exists("/dev/z90crypt"),
    "mem": sh("grep MemTotal /proc/meminfo"),
    "zdlc_or_onnx_mlir": bool(shutil.which("zdlc") or shutil.which("onnx-mlir")),
}
for m in ["torch", "numpy", "Crypto", "pydicom", "ipywidgets", "onnx", "onnxruntime", "sklearn", "PIL", "pandas"]:
    try: env[m] = getattr(importlib.import_module(m), "__version__", "ok")
    except Exception: env[m] = "MISSING"
env

{'machine': 's390x',
 'python': '3.12.15',
 'cpus': 3,
 'z_machine_type': 'machine = 9175',
 'nnpa_facility': True,
 'secure_execution_guest': 'no (/sys/firmware/uv absent)',
 'crypto_express': False,
 'mem': 'MemTotal:        7684076 kB',
 'zdlc_or_onnx_mlir': False,
 'torch': '2.11.0+cpu',
 'numpy': '2.0.2',
 'Crypto': '4.0.0',
 'pydicom': '3.0.2',
 'ipywidgets': '8.1.9',
 'onnx': '1.22.0',
 'onnxruntime': 'MISSING',
 'sklearn': '1.9.0',
 'PIL': '12.3.0',
 'pandas': '3.0.6'}

## 1 · Setup
Installs only what is missing. `ipywidgets` is needed for the demo UI — if it was just installed, **reload this browser tab** once before section 4.

In [2]:
need = {"Crypto": "pycryptodome", "pydicom": "pydicom", "ipywidgets": "ipywidgets", "kaggle": "kaggle", "onnx": "onnx"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    pkgs = " ".join(missing)
    print("installing:", pkgs)
    %pip install -q $pkgs
else:
    print("all dependencies present")

all dependencies present


In [3]:
from pathlib import Path
os.environ.setdefault("CRAI_HOME", str(Path.cwd() / "crai_data"))
sys.path.insert(0, str(Path.cwd()))
import importlib, crai
from crai import config
config.ensure_dirs()
from crai import model, crypto, attestation, guard, audit, pipeline, security_tests, bench, results, data, train, ui, explain
import torch, numpy as np, pandas as pd, json, hashlib, urllib.request
from IPython.display import HTML, display
torch.set_num_threads(os.cpu_count())
results.save("environment", {k: v for k, v in env.items()})
print("CRAI_HOME =", config.HOME)
print("MODE      =", config.MODE_LABEL)

CRAI_HOME = /home/ibm-user/confidential-radiology-ai/crai_data
MODE      = SIMULATION - no IBM Secure Execution hardware; host exclusion is NOT proven


## 2 · Frozen model + approved policy
The backbone is TorchXRayVision **`resnet50-res512-all`** (public weights, downloaded from the project's GitHub release). It is loaded into a pure-PyTorch ResNet-50 (`crai/model.py`), so torchvision is not needed on s390x.
* Before RSNA fine-tuning: the head is xrv's own **Pneumonia** output (zero-shot).
* After section 7: the head is the RSNA-trained, calibrated head, and the policy is re-approved with the new hash.

**Caveat:** the model card lists this backbone's training data as *pc-nih-rsna-siim-vin* — RSNA is in the pre-training mix, so RSNA test metrics are optimistic.

In [4]:
import urllib.request
W = config.MODELS / "xrv_resnet50_res512_all_state.pt"
if not W.exists():
    print("downloading public TorchXRayVision weights (~94 MB)…")
    urllib.request.urlretrieve(model.XRV_WEIGHTS_URL, W)
print("weights sha256:", hashlib.sha256(W.read_bytes()).hexdigest())

FT = config.MODELS / "crai_resnet50_rsna_head.pt"      # written by section 7
ZS = config.MODELS / "crai_resnet50_zeroshot.pt"
if not ZS.exists():
    model.save_frozen(model.build_zero_shot(W), ZS)
ACTIVE = FT if FT.exists() else ZS
policy = pipeline.approve_policy(ACTIVE)
print("ACTIVE MODEL :", ACTIVE.name, "(fine-tuned on RSNA)" if ACTIVE == FT else "(zero-shot, not yet fine-tuned)")
print("model_hash   :", policy["model_hash"])
print("guest_image  :", policy["guest_image_hash"], " <- sha256 over", config.GUEST_IMAGE_FILES)

weights sha256: 7724248ef6972be85a3e0d2fabe0a561634e11eb776d51b3999682c19fc596d9
ACTIVE MODEL : crai_resnet50_zeroshot.pt (zero-shot, not yet fine-tuned)
model_hash   : c64992e846e948b6c21aea2c50005a810f9cb902a428f099a29da0f409add332
guest_image  : 753e4b2ac4e2965c668f83bf58713999ed7baeea71c1a9444ab078d509fe97ab  <- sha256 over ['guest.py', 'model.py', 'crypto.py', 'guard.py', 'attestation.py', 'config.py', 'netblock.py']


## 3 · Start the attestation service + key broker (separate process)
It alone holds the KEK and the attestation signing key (in `crai_data/broker_store/`, chmod 600). The notebook process never reads them.

In [5]:
broker_proc = pipeline.start_broker()
print(pipeline.broker_call("/health"))

{'ok': True, 'mode': 'SIMULATION - no IBM Secure Execution hardware; host exclusion is NOT proven', 'token_ttl_s': 60}


## 4 · Live demo
Pick a local sample (or upload a PNG/JPG/DICOM), choose a scenario, press **Encrypt & score securely**.
The attack scenarios are real: a tampered model file, modified guest code, or altered ciphertext — watch where the pipeline refuses.

Samples: `public_sample_*` is a public NIH ChestX-ray14 image from the TorchXRayVision repository, used **only as a smoke-test image** (not RSNA, not part of any metric). `rsna_demo_*` files appear after section 7 (held-out demo split).

In [6]:
S = config.SAMPLES / "public_sample_nih_00000001_000.png"
if not S.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mlmed/torchxrayvision/master/tests/00000001_000.png", S)
print(sorted(p.name for p in config.SAMPLES.iterdir()))

['public_sample_nih_00000001_000.png']


In [7]:
ui.build(client_id="hospital-A")

In [8]:
# Same flow without widgets (works even if ipywidgets can't render)
from IPython.display import HTML, display
from PIL import Image
u8_sample = model.to_fixed_u8(Image.open(S))
res, ct = ui.run_scenario(u8_sample, "normal", "hospital-A", "public-sample")
display(HTML(ui.banner_html() + ui.pipeline_html({"encrypt": "ok", **res["stages"]}, {"encrypt": f"ciphertext {ct[:12]}…"}) + ui.result_html(res, ct)))
print(json.dumps(res["result"], indent=1))

{
 "attestation_token": "eyJlZDI1NTE5X3B1YiI6IjgyMmJlMWI1ODVlYzBlMzYxZDY3YmJlN2YxMjA2MDQ4ODY3OWRjM2IxNzZjNzY1YzEyZTdjNzhmOWM1OTk3MzMiLCJleHAiOjE3OTIyMjEzOTAsImd1ZXN0X2ltYWdlX2hhc2giOiI3NTNlNGIyYWM0ZTI5NjVjNjY4ZjgzYmY1ODcxMzk5OWVkN2JhZWVhNzFjMWE5NDQ0YWIwNzhkNTA5ZmU5N2FiIiwiaWF0IjoxNzkyMjIxMzMwLCJpc3MiOiJjcmFpLWF0dGVzdGF0aW9uLXNlcnZpY2UgKHNpbXVsYXRlZCkiLCJtb2RlbF9oYXNoIjoiYzY0OTkyZTg0NmU5NDhiNmMyMWFlYTJjNTAwMDVhODEwZjljYjkwMmE0MjhmMDk5YTI5ZGEwZjQwOWFkZDMzMiIsIm5vbmNlIjoiMDE1YTNiNGM4Y2IzOWU5MzIzZWM4MzNmOGQ1OTAyMjUxNzA1NWI5OGMwNDQyMGQyM2M5ODZkOTc2YTZjMWZjNyIsIngyNTUxOV9wdWIiOiIzMDJhMzAwNTA2MDMyYjY1NmUwMzIxMDA5NGEyMGY3MWU2NzJmMzNkMzMyYWExZWMxNmZjMzZkZWQ2MjE5NTJlYzExNjkzNmZmNzJmMzBkYzgwMTYzYTFmIn0.YtNjfezsbqSRkCkdGl3B4wRHZUgr3g3PbWh_-4FU3Ngi5FpISCbe03K8Lc_GLp3ohUraMrStDqUnaw3fITFcCg",
 "payload": {
  "issued_at": 1792221332,
  "model_hash": "c64992e846e948b6c21aea2c50005a810f9cb902a428f099a29da0f409add332",
  "pneumonia_probability": "0.0067",
  "schema": "crai.result.v1",
  "study_hash": "d

## 4b · Plain-language explanations with IBM Granite (outside the guest)
The demo UI above has a checkbox that adds an explanation of every security decision. A fixed, hand-written explanation is always shown (authoritative). **IBM Granite 4.0 350M** (the GGUF file in the Jupyter root, run with llama.cpp on s390x) rewrites it in plainer words for a hospital manager.
Privacy boundary: Granite runs on the host side and only receives the decision code, stage and failed check names, **never pixels, keys, ids, hashes or the probability**. It explains security decisions; it does not read the X-ray or give medical advice. A **faithfulness check** withholds Granite's wording if it claims something happened that did not (e.g. "the key was released" on a refusal) or drifts from the facts; the small base model does this sometimes, and we show it.

In [9]:
from crai import explain
ex = ui.explainer()
print("Granite available:", ex.available, "|", ex.path)
demo_events = [explain.event_from_result(res),
               {"decision": "refused", "reason": "guest_image_hash_mismatch", "stage": "attestation", "failed_checks": []},
               {"decision": "refused", "reason": "study_hash_mismatch", "stage": "key_release", "failed_checks": []}]
outs = [ex.explain(e) for e in demo_events]
for e, o in zip(demo_events, outs):
    display(HTML(f"<b>{e['reason']}</b>" + ui.explanation_html(o)))
if ex.available:
    results.save("granite_explainer", {"model": outs[0]["model"], "load_ms": round(ex.load_ms),
                 "median_generation_ms": float(np.median([o["granite_ms"] for o in outs])), "events_explained": len(outs),
                 "withheld_by_faithfulness_check": sum(o["granite"] is None for o in outs)},
                 note="host side, outside the guest; input = decision code + stage only; wording only, not authoritative")

Granite available: True | /home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


## 5 · Security tests
Each row sets up one attack on the real code path and passes only if the system refuses as expected (≈1–3 min: several guest runs).

In [10]:
import pandas as pd
rows = security_tests.run_all(u8_sample)
sec_df = pd.DataFrame(rows)
results.save("security_tests", {r["#"]: r["result"] for r in rows})
pipeline.limiter.state.clear()
sec_df

,#,test,attack,expected,observed,result
0,1,Wrong model hash,1 byte flipped in model file,model_hash_mismatch,model_hash_mismatch,PASS
1,2,Wrong guest image hash,1 line added to guest.py,guest_image_hash_mismatch,guest_image_hash_mismatch,PASS
2,3,Truncated ciphertext,last 100 bytes removed,CiphertextError: malformed/truncated,malformed or truncated ciphertext,PASS
3,4,Invalid AES-GCM tag,last tag byte flipped,CiphertextError: invalid tag,authentication failed (invalid AES-GCM tag),PASS
4,5,Expired attestation token,valid token used after its 2 s lifetime,token_expired,token_expired,PASS
5,6,Study-hash mismatch,1 byte of vault ciphertext changed,study_hash_mismatch,study_hash_mismatch,PASS
6,7,Unexpected output field,extra 'image_png' / 'patient_id' fields,SchemaError on both,envelope: rejected; payload: rejected,PASS
7,E1,Replay,re-send evidence with a used nonce,nonce_unknown_or_used,nonce_unknown_or_used,PASS
8,E2,Rate limit,20 requests in 2 s from one client,"5 allowed, 15 refused","5 allowed, 15 refused",PASS
9,E3,Audit log tampering,line 80 edited in a copy of the log,chain breaks at line 80,"ok=False, first bad line=80",PASS


## 6 · Secure path vs clear path, latency, Telum
* **Parity:** the same studies are scored in the clear (in-process) and through the full secure path. The secure output is rounded to 4 decimals, so the chosen tolerance is **1e-4**.
* Studies used: RSNA held-out demo studies if section 7 has been run, otherwise the single public sample (labelled as such).

In [11]:
demo_files = sorted(config.SAMPLES.glob("rsna_demo_*.npy"))
if demo_files:
    parity_u8, parity_src = [np.load(p) for p in demo_files], "rsna-demo"
else:
    parity_u8, parity_src = [u8_sample], "public-sample (not RSNA)"
net = model.load_frozen_bytes(ACTIVE.read_bytes())
summary, prow = bench.secure_vs_clear(net, parity_u8, parity_src)
summary["studies_source"], summary["model"] = parity_src, ACTIVE.name
results.save("secure_vs_clear", summary)
display(pd.DataFrame(prow)); summary

,study_id,clear,secure,verified,wall_ms,measure_ms,attest_release_ms,model_load_ms,decrypt_ms,infer_ms,sign_ms
0,300dc2f29b046c1d,0.006716,0.0067,True,2414.4,53.7,16.4,262.1,4.1,994.8,0.9


{'studies': 1,
 'verified': 1,
 'mean_abs_diff': 1.551981940865494e-05,
 'max_abs_diff': 1.551981940865494e-05,
 'tolerance': 0.0001,
 'within_tolerance': True,
 'studies_source': 'public-sample (not RSNA)',
 'model': 'crai_resnet50_zeroshot.pt'}

In [12]:
lat = bench.clear_latency(net, u8_sample, runs=5)
results.save("latency_clear", lat, note="PyTorch CPU, batch 1, 512x512")
sec_ms = [r["wall_ms"] for r in prow]
results.save("latency_secure", {"n": len(sec_ms), "median_wall_ms": float(np.median(sec_ms)),
             "guest_breakdown_ms_last": prow[-1] and {k: v for k, v in prow[-1].items() if k.endswith("_ms")}},
             note="includes process start, model load, attestation, decrypt, inference, signing")
lat, results.load()["latency_secure"]["value"]

({'runs': 5,
  'median_ms': 1001.5482610033359,
  'min_ms': 991.2005459991633,
  'max_ms': 1003.2023739986471,
  'torch_threads': 3,
  'machine': 's390x'},
 {'n': 1,
  'median_wall_ms': 2414.4,
  'guest_breakdown_ms_last': {'wall_ms': 2414.4,
   'measure_ms': 53.7,
   'attest_release_ms': 16.4,
   'model_load_ms': 262.1,
   'decrypt_ms': 4.1,
   'infer_ms': 994.8,
   'sign_ms': 0.9}})

In [13]:
# ONNX operator inventory of the frozen model, and Telum (NNPA) placement status
onnx_path = bench.export_onnx(net, config.MODELS / (ACTIVE.stem + ".onnx"))
ops = bench.onnx_operator_inventory(onnx_path)
results.save("onnx_operators", ops)
if env["zdlc_or_onnx_mlir"]:
    print("zDLC/onnx-mlir found: compile with --maccel=NNPA and record which ops were offloaded.")
else:
    results.save("telum_operator_placement", "not measured",
                 note="per-operator placement not enumerated (no zDLC/onnx-mlir here); whole-model NNPA speed-up is measured in section 6b")
print(ops)
print("Telum operator placement:", results.load()["telum_operator_placement"]["value"])

{'Conv': 53, 'Relu': 49, 'Add': 16, 'MaxPool': 1, 'GlobalAveragePool': 1, 'Flatten': 1, 'Gemm': 1, 'Div': 1, 'Sigmoid': 1, 'Constant': 1, 'Squeeze': 1}
Telum operator placement: not measured


## 6b · Telum on-chip AI accelerator (NNPA): measured
Our frozen ResNet-50 is rebuilt in **TensorFlow/Keras with the same weights** and run twice, each in a fresh process: with IBM's **ibm-zdnn-plugin** (TensorFlow kernels that call the zDNN library, which uses the Telum accelerator's NNPA instructions), and with that plugin hidden from TensorFlow for that process only (nothing on disk changes). Same inputs, same number of runs. The secure guest itself still runs PyTorch; this is a measurement of the model on Telum.

In [14]:
from crai import telum
TW = telum.export_weights(net, config.MODELS / "telum_weights.npz")
tel_u8 = np.stack(parity_u8[:8])
np.save(config.MODELS / "telum_inputs.npy", tel_u8)
torch_p = np.array([model.predict_u8(net, u) for u in tel_u8])
tel = {m: telum.run_mode(m, TW, config.MODELS / "telum_inputs.npy", config.MODELS / f"telum_{m}.json") for m in ["nnpa", "cpu"]}
summ = {"model": ACTIVE.name, "images": int(len(tel_u8)), "images_source": parity_src}
for bs in (1, 8):
    c, n = tel["cpu"][f"batch{bs}"]["median_ms"], tel["nnpa"][f"batch{bs}"]["median_ms"]
    summ[f"batch{bs}_tf_cpu_ms"], summ[f"batch{bs}_tf_nnpa_ms"], summ[f"batch{bs}_speedup"] = round(c, 1), round(n, 1), round(c / n, 2)
summ["max_abs_diff_tf_nnpa_vs_pytorch"] = float(np.abs(np.array(tel["nnpa"]["probs"]) - torch_p).max())
summ["max_abs_diff_tf_cpu_vs_pytorch"] = float(np.abs(np.array(tel["cpu"]["probs"]) - torch_p).max())
results.save("telum_benchmark", summ, note="TF + ibm-zdnn-plugin vs same TF with plugin hidden; same weights; median of 7 runs")
display(pd.DataFrame({m: {"batch 1 median ms": tel[m]["batch1"]["median_ms"], "batch 8 median ms": tel[m]["batch8"]["median_ms"],
                          "images/s (batch 8)": tel[m]["batch8"]["images_per_s"]} for m in ["cpu", "nnpa"]}).round(1))
summ

,cpu,nnpa
batch 1 median ms,318.1,95.3
batch 8 median ms,2541.9,1011.1
images/s (batch 8),3.1,7.9


{'model': 'crai_resnet50_zeroshot.pt',
 'images': 1,
 'images_source': 'public-sample (not RSNA)',
 'batch1_tf_cpu_ms': 318.1,
 'batch1_tf_nnpa_ms': 95.3,
 'batch1_speedup': 3.34,
 'batch8_tf_cpu_ms': 2541.9,
 'batch8_tf_nnpa_ms': 1011.1,
 'batch8_speedup': 2.51,
 'max_abs_diff_tf_nnpa_vs_pytorch': 0.00037549296393990517,
 'max_abs_diff_tf_cpu_vs_pytorch': 3.259629011154175e-09}

## 7 · RSNA data, patient-level split, head fine-tuning, evaluation
**Access requirements (Kaggle):** the RSNA Pneumonia Detection Challenge data is not directly downloadable. You need (1) a Kaggle account, (2) to accept the competition rules at `kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules`, (3) your own API token placed **by you** on this server in `~/.kaggle/kaggle.json` (`chmod 600`) — do not paste it into this notebook. Alternatively download the zip yourself and unzip it into `crai_data/data/rsna/`.

Every cell below checks what is present and skips cleanly if the data is not there yet.

In [15]:
data.status()

{'kaggle_credentials_present': False,
 'labels_csv': False,
 'train_dicoms': 0,
 'splits_csv': False,
 'cached_npy': 0}

In [16]:
# Download (~3.5-4 GB) with the official Kaggle CLI using YOUR token on this server
st = data.status()
if st["train_dicoms"] == 0 and st["kaggle_credentials_present"]:
    print(data.download())
elif st["train_dicoms"] == 0:
    print("SKIPPED: no Kaggle token on this server and no data in", data.RSNA)
else:
    print("data present:", st)

SKIPPED: no Kaggle token on this server and no data in /home/ibm-user/confidential-radiology-ai/crai_data/data/rsna


In [17]:
# Patient-level split (frozen to splits.csv): ~70/15/15 + 20 held-out demo studies (10 pos / 10 neg)
if data.LABELS.exists():
    df = data.load_splits() if data.SPLITS.exists() else data.make_splits(seed=42)
    tbl = df.groupby("split").Target.agg(["count", "sum", "mean"]).rename(columns={"sum": "positives", "mean": "prevalence"})
    results.save("data_split", {"patients": int(len(df)), **{s: {"n": int(r["count"]), "pos": int(r["positives"])} for s, r in tbl.iterrows()},
                                "splits_sha256": hashlib.sha256(data.SPLITS.read_bytes()).hexdigest()[:16]})
    display(tbl)
    assert not set(df[df.split=="train"].patientId) & set(df[df.split!="train"].patientId), "patient leakage!"
else:
    print("SKIPPED: labels not present")

SKIPPED: labels not present


In [18]:
# Subset sizes for the 3-vCPU server. Stratified subsets keep the class ratio. None = whole split.
N_TRAIN, N_VAL, N_TEST = 3000, 800, 1500
if data.LABELS.exists():
    tr = data.subsample(df, "train", N_TRAIN); va = data.subsample(df, "val", N_VAL); te = data.subsample(df, "test", N_TEST)
    dm = df[df.split == "demo"]
    need = pd.concat([tr, va, te, dm]).patientId
    data.cache_images(need, workers=os.cpu_count())
    s_per_img = train.measure_throughput(net)
    print(f"{s_per_img:.2f} s/image -> feature extraction for {len(tr)+len(va)+len(te)} images ≈ {s_per_img*(len(tr)+len(va)+len(te))/60:.0f} min")

In [19]:
# Feature extraction (resumable: re-run this cell if it is interrupted)
if data.LABELS.exists():
    prog = lambda d, n: print(f"\r{d}/{n}", end="")
    Xtr = train.extract_features(net, tr.patientId, "train", progress=prog); ytr = tr.Target.values
    Xva = train.extract_features(net, va.patientId, "val", progress=prog);   yva = va.Target.values
    Xte = train.extract_features(net, te.patientId, "test", progress=prog);  yte = te.Target.values
    print("\nfeatures:", Xtr.shape, Xva.shape, Xte.shape)

In [20]:
# Train head on TRAIN, choose C + calibrate + pick threshold on VAL, report on TEST once.
if data.LABELS.exists():
    zs = model.build_zero_shot(W)
    fw, fb = zs.backbone.fc.weight[model.PNEUMONIA_IDX].detach().numpy(), float(zs.backbone.fc.bias[model.PNEUMONIA_IDX])
    p_zs_va, p_zs_te = train.sigmoid(Xva @ fw + fb), train.sigmoid(Xte @ fw + fb)
    zs_thr = train.threshold_for_sensitivity(yva, p_zs_va, 0.90)
    results.save("eval_zero_shot_test", train.summarize(yte, p_zs_te, zs_thr),
                 note=f"xrv Pneumonia unit; test subset n={len(yte)}; RSNA in backbone pre-training (optimistic)")

    w, b, C = train.fit_head(Xtr, ytr, Xva, yva)                    # class_weight='balanced' + Platt on val
    p_va, p_te = train.sigmoid(Xva @ w + b), train.sigmoid(Xte @ w + b)
    thr = train.threshold_for_sensitivity(yva, p_va, 0.90)          # fixed on val
    results.save("eval_finetuned_test", {**train.summarize(yte, p_te, thr), "C": C, "n_train": int(len(ytr))},
                 note=f"head-only fine-tune; threshold for val sensitivity>=0.90; RSNA in backbone pre-training (optimistic)")
    display(pd.DataFrame({"zero-shot": results.load()["eval_zero_shot_test"]["value"],
                          "fine-tuned head": results.load()["eval_finetuned_test"]["value"]}))

In [21]:
# Freeze the fine-tuned model, re-approve the policy with its hash, export demo studies for the UI
if data.LABELS.exists():
    ft = train.build_finetuned(W, w, b)
    print("fine-tuned model_hash:", model.save_frozen(ft, FT))
    ACTIVE, net = FT, ft
    policy = pipeline.approve_policy(ACTIVE)
    for i, pid in enumerate(dm.patientId):                       # no patient ids in file names
        np.save(config.SAMPLES / f"rsna_demo_{i:02d}.npy", data.load_u8(pid))
    print("policy now approves:", policy["model_hash"][:16], "| demo samples:", len(dm))
    print("Re-run sections 4-6 to demo and measure with the fine-tuned model.")

## 8 · Results (only what was measured here) and scope

In [22]:
display(HTML(results.table_html()))
ok, bad = pipeline.audit.verify_chain()
print(f"audit log: {len(pipeline.audit.records())} events, hash chain {'intact' if ok else 'BROKEN at line %s' % bad}")

metric,value,measured,note
environment,"{""machine"": ""s390x"", ""python"": ""3.12.15"", ""cpus"": 3, ""z_machine_type"": ""machine = 9175"", ""nnpa_facility"": true, ""secure_execution_guest"": ""no (/sys/firmware/uv absent)"", ""crypto_express"": false, ""mem"": ""MemTotal: …",2026-10-17 07:15:28 (s390x),
security_tests,"{""1"": ""PASS"", ""2"": ""PASS"", ""3"": ""PASS"", ""4"": ""PASS"", ""5"": ""PASS"", ""6"": ""PASS"", ""7"": ""PASS"", ""E1"": ""PASS"", ""E2"": ""PASS"", ""E3"": ""PASS"", ""E4"": ""PASS""}",2026-10-17 07:16:04 (s390x),
secure_vs_clear,"{""studies"": 1, ""verified"": 1, ""mean_abs_diff"": 1.551981940865494e-05, ""max_abs_diff"": 1.551981940865494e-05, ""tolerance"": 0.0001, ""within_tolerance"": true, ""studies_source"": ""public-sample (not RSNA)"", ""model"": ""crai_res…",2026-10-17 07:16:08 (s390x),
latency_clear,"{""runs"": 5, ""median_ms"": 1001.5482610033359, ""min_ms"": 991.2005459991633, ""max_ms"": 1003.2023739986471, ""torch_threads"": 3, ""machine"": ""s390x""}",2026-10-17 07:16:14 (s390x),"PyTorch CPU, batch 1, 512x512"
latency_secure,"{""n"": 1, ""median_wall_ms"": 2414.4, ""guest_breakdown_ms_last"": {""wall_ms"": 2414.4, ""measure_ms"": 53.7, ""attest_release_ms"": 16.4, ""model_load_ms"": 262.1, ""decrypt_ms"": 4.1, ""infer_ms"": 994.8, ""sign_ms"": 0.9}}",2026-10-17 07:16:14 (s390x),"includes process start, model load, attestation, decrypt, inference, signing"
onnx_operators,"{""Conv"": 53, ""Relu"": 49, ""Add"": 16, ""MaxPool"": 1, ""GlobalAveragePool"": 1, ""Flatten"": 1, ""Gemm"": 1, ""Div"": 1, ""Sigmoid"": 1, ""Constant"": 1, ""Squeeze"": 1}",2026-10-17 07:16:16 (s390x),
telum_operator_placement,"""not measured""",2026-10-17 07:16:16 (s390x),per-operator placement not enumerated (no zDLC/onnx-mlir here); whole-model NNPA speed-up is measured in section 6b
telum_benchmark,"{""model"": ""crai_resnet50_zeroshot.pt"", ""images"": 1, ""images_source"": ""public-sample (not RSNA)"", ""batch1_tf_cpu_ms"": 318.1, ""batch1_tf_nnpa_ms"": 95.3, ""batch1_speedup"": 3.34, ""batch8_tf_cpu_ms"": 2541.9, ""batch8_tf_nnpa_m…",2026-10-17 07:17:02 (s390x),TF + ibm-zdnn-plugin vs same TF with plugin hidden; same weights; median of 7 runs
granite_explainer,"{""model"": ""IBM Granite 4.0 350M base (granite-4.0-h-350m-base-be.f16.gguf, llama.cpp)"", ""load_ms"": 2578, ""median_generation_ms"": 5115.5, ""events_explained"": 3, ""withheld_by_faithfulness_check"": 1}",2026-10-17 07:15:54 (s390x),"host side, outside the guest; input = decision code + stage only; wording only, not authoritative"
data_split,not measured,,


audit log: 164 events, hash chain intact


### Side-channel scope (what is and is not claimed)
**Implemented:** fixed 512×512 input shape for every study · fixed-length responses (probability is always a 6-character string) · the output guard does the same work whatever the score · AES-GCM and Ed25519 from PyCryptodome (constant-time behaviour not independently verified here) · guest outbound network blocked except the broker (in-process guard) · decrypted buffer and input tensor zeroed after inference.

**Not claimed:** protection against cache-timing, speculative-execution, power or memory-remanence attacks; complete erasure of every in-memory copy (Python/allocator copies may remain); host exclusion (needs real IBM Secure Execution); clinical validity; regulatory compliance.

### Moving to real IBM Z security features
| This prototype | Production on IBM Z |
|---|---|
| guest = separate process | IBM Secure Execution KVM guest (`genprotimg`) |
| self-reported measurement | Ultravisor-signed attestation (`pvattest`) verified by a Trustee attestation service |
| KEK file in `broker_store/` | Trustee key broker, KEK in Crypto Express (HSM); protected keys (`paes`) via CPACF |
| PyCryptodome AES (no CPACF claim) | OpenSSL / libica using CPACF |
| in-process egress guard | SE image with network policy allowing only the broker |

In [23]:
# Stop the broker when you are done (it is restarted by section 3)
# broker_proc.terminate() if broker_proc else None